# Model notebook

This notebook will deal with training and predicting on the dataset that was created in the EDA notebook. The current notebook will be divided into several sections:

1. Setting up the docker container for LightGBM purposes
2. Training and predicting on sklearn Linear Regression as the base model
3. Training and predicting on sklearn Random Forest as the first target model
4. Training and predicting on lightgbm trees as the second target model
5. Showing how customer-level predictions can be used to calculate an uplift on the offer

Furthermore, the shortcomings of the approach will be discussed, with possible solutions to check for the future.

In [1]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
from sagemaker.predictor import csv_serializer
import pandas as pd
import boto3
import sagemaker
from sagemaker import get_execution_role
from processing.clean_data import *
from sagemaker.sklearn.estimator import SKLearn
from sagemaker.tuner import IntegerParameter, ContinuousParameter, HyperparameterTuner

## 1. Docker container for LightGBM

In [2]:
# I need to set up parameters for the container with LightGBM
# as well as for training with both Scikit-lears and LightGBM
ecr_namespace = "udacity-capstone-project/"
prefix = "lightgbm-container"

# all the necessary parameters
ecr_repository_name = ecr_namespace + prefix
role = get_execution_role()
account_id = role.split(":")[4]
region = boto3.Session().region_name
sess = sagemaker.session.Session()
bucket = sess.default_bucket()

print(account_id)
print(region)
print(role)
print(bucket)

413371515459
us-east-1
arn:aws:iam::413371515459:role/service-role/AmazonSageMaker-ExecutionRole-20210918T134320
sagemaker-us-east-1-413371515459


The following cell has to run to make the building and pushing of the container possible.
The .sh file needs to have proper rights.

In [3]:
%%sh
chmod +x ./docker/build_and_push.sh

The next cell is building and pushing the docker container, so that it can be used in Sagemaker. The basic structure of the container will contain the Dockerfile with configuration, as well as the program containing train, serve and predictor.py files - those will be used by Sagemaker to run proper training and make inferences.

The reason for creating the container in the first place is that LightGBM is not supported off-the-shelf by Sagemaker. As a result, a container with necessary packages + a program with training and predicting scripts need to be created.

In [4]:
%%capture captured
! ./docker/build_and_push.sh $account_id $region $ecr_repository_name

In [5]:
# we can see if the build and push were successful
captured()


Step 1/15 : FROM ubuntu:16.04
 ---> b6f507652425
Step 2/15 : LABEL maintainer="Amazon AI"
 ---> Using cache
 ---> abc3211d0eff
Step 3/15 : ARG PYTHON=python3
 ---> Using cache
 ---> eef483690b73
Step 4/15 : ARG PYTHON_PIP=python3-pip
 ---> Using cache
 ---> 31cc41690bb2
Step 5/15 : ARG PIP=pip3
 ---> Using cache
 ---> c59ef5f213ae
Step 6/15 : ARG PYTHON_VERSION=3.6.6
 ---> Using cache
 ---> 8e532f1b8885
Step 7/15 : RUN apt-get update && apt-get install -y --no-install-recommends software-properties-common &&     add-apt-repository ppa:deadsnakes/ppa -y &&     apt-get update && apt-get install -y --no-install-recommends         build-essential         ca-certificates         curl         wget         nginx         git         libopencv-dev         openssh-client         openssh-server         vim         zlib1g-dev &&     rm -rf /var/lib/apt/lists/*
 ---> Using cache
 ---> 0cd39700a096
Step 8/15 : RUN wget https://www.python.org/ftp/python/$PYTHON_VERSION/Python-$PYTHON_VERSION.tgz && 

In [6]:
# this is where my container will be located
container_image_uri = "{0}.dkr.ecr.{1}.amazonaws.com/{2}:latest".format(
    account_id, region, ecr_repository_name
)
print(container_image_uri)

413371515459.dkr.ecr.us-east-1.amazonaws.com/udacity-capstone-project/lightgbm-container:latest


## 2. Preparing data

In [7]:
# a function to split the data and, if necessary, standardize it beforehand
def split_data(transaction_df, feature_list, test_size= 0.3, random_state=1, standardize=False):
    '''
    The purpose of this function is to split the dataset into train and test subsets
    with additional possibility of standardizing the data beforehand for LR
    
    INPUT: 
        df: Dataframe of all credit card transaction data
        test_size: The decimal fraction of data that should be training data
        Random seed for shuffling and reproducibility, default = 1
    OUTPUT:
        train_df: Dataframe containing train data
        test_df: Dataframe containing test data
    '''
    
    # standardize should be True for Linear Regression model
    # we only want to standardize continuous, non-binary features
    if standardize:
        cols_to_scale = ['reward', 'difficulty', 'age', 'income', 'year']
        scale = StandardScaler()
        transaction_df_scaled = transaction_df.copy()
        transaction_df_scaled[cols_to_scale] = scale.fit_transform(transaction_df_scaled[cols_to_scale])
        train_df, test_df = train_test_split(transaction_df_scaled[feature_list], test_size=test_size, random_state = random_state)
    else:
        train_df, test_df = train_test_split(transaction_df[feature_list], test_size=test_size, random_state = random_state)
    return train_df, test_df

In [8]:
# loading (or creating) the data
name = 'data/finaldata'
my_file = Path(name + ".csv")

# if the dataset is not present I need to generate some intermediate datasets first
if not my_file.is_file():
    portfolio = pd.read_json('data/portfolio.json', orient='records', lines=True)
    profile = pd.read_json('data/profile.json', orient='records', lines=True)
    transcript = pd.read_json('data/transcript.json', orient='records', lines=True)

    qry = '''
    select distinct
        r.*, v.time as time_viewed, c.time as time_completed
    from
        received r
        left join viewed v on r.time <= v.time and r.person = v.person and r.offer_id = v.offer_id
        left join completed c on r.time <= c.time and r.person = c.person and r.offer_id = c.offer_id
    '''
    
    profile_clean = clean_profile(profile)
    portfolio_clean = clean_portfolio(portfolio)
    transcript_clean = clean_transcript(transcript)

    df_offer_final, df_transaction_final = create_final_offer_and_trans_dataset(transcript_clean, portfolio_clean, qry)
    df = get_dataset(df_offer_final, df_transaction_final, portfolio_clean, profile_clean, name, True)
else:
    df = pd.read_csv(name + ".csv")
    
# we will have two sets of features, one with OHE columns for the linear regression
# the other for LightGBM which does not require OHE, only to mark which columns are categorical in the training script
features_lgbm = ['reward', 'difficulty', 'channels_int', 'offer_type_int','gender',
                 'age', 'income', 'year', 'hourly_sum']
features_sklin = ['reward', 'difficulty', 'email', 'mobile','social', 'web', 'bogo', 
                  'discount', 'informational','gender', 'age', 'income', 'year', 'hourly_sum']

## 3. SKLearn Linear Regression

I will start with a simple Linear Regression algorithm that is available in sklearn. Sagemaker already offers off-the-shelf estimators that work well regarding scikit-learn.

All that I need to do is to prepare the data, set up an estimator, and fit it. Then, I will use the metric of my choice to check the results.

In the next cells I specify the estimator and fit it. It is a simple linear regression on standardized data

In [9]:
train_name = 'data/starbucks_train_lr.csv'

# splitting the data and standardizing it in the meantime
Xtrain_sk, Xtest_sk = split_data(df, features_sklin, standardize=True)

# not necessary, but in good practice - getting the first column of the dataframe as labels
# the other columns will be then features
Xtrain_sk = pd.concat([Xtrain_sk.iloc[:,-1], Xtrain_sk.iloc[:,:-1]], axis=1)
Xtest_sk = pd.concat([Xtest_sk.iloc[:,-1], Xtest_sk.iloc[:,:-1]], axis=1)

Xtrain_sk.to_csv(train_name, index=False)

# uploading the training data to S3
train_location_sk = sess.upload_data(path=train_name, 
                                  bucket=bucket,
                                  key_prefix='sklearn')

# defining the channel, the framework version and the path to the script
s3_input = {'train': train_location_sk}
FRAMEWORK_VERSION = "0.23-1"
script_path = "skscripts/sk-train-lr.py"

# setting up an estimator
sklearn = SKLearn(
    entry_point=script_path,
    framework_version=FRAMEWORK_VERSION,
    instance_type="ml.c4.xlarge",
    role=role,
    sagemaker_session=sess
)

# and fitting it
sklearn.fit(s3_input)

2022-01-10 17:05:05 Starting - Starting the training job...
2022-01-10 17:05:10 Starting - Launching requested ML instancesProfilerReport-1641834304: InProgress
......
2022-01-10 17:06:26 Starting - Preparing the instances for training.........
2022-01-10 17:07:46 Downloading - Downloading input data...
2022-01-10 17:08:26 Training - Downloading the training image..2022-01-10 17:08:42,678 sagemaker-containers INFO     Imported framework sagemaker_sklearn_container.training
2022-01-10 17:08:42,680 sagemaker-training-toolkit INFO     No GPUs detected (normal if no gpus installed)
2022-01-10 17:08:42,692 sagemaker_sklearn_container.training INFO     Invoking user training script.

2022-01-10 17:09:07 Uploading - Uploading generated training model2022-01-10 17:08:57,049 sagemaker-training-toolkit INFO     No GPUs detected (normal if no gpus installed)
2022-01-10 17:08:57,061 sagemaker-training-toolkit INFO     No GPUs detected (normal if no gpus installed)
2022-01-10 17:08:57,075 sagemaker

Now that the model is fit, I can deploy it and predict on the test dataset.

In [10]:
# setting up a predictor
predictor = sklearn.deploy(initial_instance_count=1, instance_type="ml.m5.xlarge")

------!

In [11]:
# now that the predictor has been set up, I can check the results
preds = predictor.predict(Xtest_sk.iloc[:,1:])
print(preds)
print(Xtest_sk.iloc[:,0].values)
rmse = mean_squared_error(Xtest_sk.iloc[:,0], preds, squared=False)
rmse

[2.19970164 2.93108392 1.15563333 ... 1.85645746 2.03864073 0.4573302 ]
[ 0.06125     0.          0.         ...  0.16918919 12.22
  0.24969479]


8.425908376009604

RMSE seems to be quite high when we are predicting the hourly spend per customer. 

As we can see in the first printouts, the predicted values of dollars spent per hour are very different from the actual ones - most of the time we are overpredicting the actual spend.

When thinking about it more, it might be the fault of the outliers that we could see in the EDA notebook. The problem is mainly related to customers who take no time to realize an offer (time viewed is equal to completion time, and as such the hourly spend is very big). I will now try to make another fit but excluding top 5% results of the data regarding the hourly spend.

In [12]:
# cleaning up
predictor.delete_endpoint()

In [13]:
train_name = 'data/starbucks_train_lr_noout.csv'

# splitting the data and standardizing it in the meantime (only up to 95th percentile)
Xtrain_sk, Xtest_sk = split_data(df[df['hourly_sum'] <= df['hourly_sum'].quantile(0.95)],
                                 features_sklin, standardize=True)

# not necessary, but in good practice - getting the first column of the dataframe as labels
# the other columns will be then features
Xtrain_sk = pd.concat([Xtrain_sk.iloc[:,-1], Xtrain_sk.iloc[:,:-1]], axis=1)
Xtest_sk = pd.concat([Xtest_sk.iloc[:,-1], Xtest_sk.iloc[:,:-1]], axis=1)

Xtrain_sk.to_csv(train_name, index=False)

# uploading the training data to S3
train_location_sk = sess.upload_data(path=train_name, 
                                  bucket=bucket,
                                  key_prefix='sklearn')

# defining the channel, the framework version and the path to the script
s3_input = {'train': train_location_sk}
FRAMEWORK_VERSION = "0.23-1"
script_path = "skscripts/sk-train-lr.py"

# setting up an estimator
sklearn = SKLearn(
    entry_point=script_path,
    framework_version=FRAMEWORK_VERSION,
    instance_type="ml.c4.xlarge",
    role=role,
    sagemaker_session=sess
)

In [14]:
# and fitting
sklearn.fit(s3_input)

2022-01-10 17:12:51 Starting - Starting the training job...
2022-01-10 17:13:17 Starting - Launching requested ML instancesProfilerReport-1641834771: InProgress
......
2022-01-10 17:14:17 Starting - Preparing the instances for training.........
2022-01-10 17:15:53 Downloading - Downloading input data......
2022-01-10 17:16:47 Training - Training image download completed. Training in progress..2022-01-10 17:16:47,730 sagemaker-containers INFO     Imported framework sagemaker_sklearn_container.training
2022-01-10 17:16:47,732 sagemaker-training-toolkit INFO     No GPUs detected (normal if no gpus installed)
2022-01-10 17:16:47,743 sagemaker_sklearn_container.training INFO     Invoking user training script.
2022-01-10 17:16:48,199 sagemaker-training-toolkit INFO     No GPUs detected (normal if no gpus installed)
2022-01-10 17:16:51,228 sagemaker-training-toolkit INFO     No GPUs detected (normal if no gpus installed)
2022-01-10 17:16:51,242 sagemaker-training-toolkit INFO     No GPUs dete

In [15]:
# setting up a predictor
predictor = sklearn.deploy(initial_instance_count=1, instance_type="ml.m5.xlarge")

-----!

In [16]:
# now that the predictor has been set up, I can check the results
preds = predictor.predict(Xtest_sk.iloc[:,1:])
print(preds)
print(Xtest_sk.iloc[:,0].values)
rmse = mean_squared_error(Xtest_sk.iloc[:,0], preds, squared=False)
rmse

[0.61001275 0.36189618 0.20766941 ... 0.18019369 0.82710372 0.51638605]
[0.51974684 0.26343137 0.05665734 ... 0.12033708 0.52651163 0.        ]


0.5331752843832804

RMSE seems to be much lower if we exclude the outliers 

As we can see in the first printouts, the predicted values of dollars spent per hour got much closer to the actual ones, while previously we were heavily overpredicting the hourly spend.

The outliers did play a role in increasing the value of predictions. But since the outliers are not just data quirks and are valid data points, maybe there is a different way of approaching it and still getting good results? Or maybe a separate model for the outliers (an outlier is usually someone who completes the offer almost immediately after viewing it) would be good? For the time being, let's delete the endpoint and go for random forests

In [17]:
# cleaning up
predictor.delete_endpoint()

## 4. Random Forests

As specified in the project proposal, I will first go for the random forest option and see how it performs. In order to do that, I can use the same off-the-shelf SKLearn estimator that I used for linear regression. There will also be no need for standardization of data as I am using a tree-based model. Lastly, the features used will be the same as in LightGBM, and the dataset will be cut at 95th percentile as with linear regression.

I will not wait anymore for tuning hyperparameters - the way I will handle the random forest model will be first building a tuner, fitting it, and then deploying the best estimator from the tuning job.

In [18]:
train_name = 'data/starbucks_train_rf.csv'
test_name = 'data/starbucks_test_rf.csv'

# splitting the data, but no need to standardize as I am using a tree based method
Xtrain_rf, Xtest_rf = split_data(df[df['hourly_sum'] <= df['hourly_sum'].quantile(0.95)],
                                 features_lgbm, standardize=False)

# not necessary, but in good practice - getting the first column of the dataframe as labels
# the other columns will be then features
Xtrain_rf = pd.concat([Xtrain_rf.iloc[:,-1], Xtrain_rf.iloc[:,:-1]], axis=1)
Xtest_rf = pd.concat([Xtest_rf.iloc[:,-1], Xtest_rf.iloc[:,:-1]], axis=1)

Xtrain_rf.to_csv(train_name, index=False)
Xtest_rf.to_csv(test_name, index=False)

train_location_rf = sess.upload_data(path=train_name, 
                                  bucket=bucket,
                                  key_prefix='sklearn-rf')

test_location_rf = sess.upload_data(path=test_name, 
                                 bucket=bucket,
                                 key_prefix='sklearn-rf')


# defining the channel, the framework version and the path to the script
s3_input = {'train': train_location_rf, 'test': test_location_rf}
FRAMEWORK_VERSION = "0.23-1"
script_path = "skscripts/sk-train-rf.py"

# setting up an estimator to pass to the tuner
sklearn = SKLearn(
    entry_point=script_path,
    framework_version=FRAMEWORK_VERSION,
    instance_type="ml.c4.xlarge",
    role=role,
    sagemaker_session=sess
)


# setting up tuner parameters and metrics
tuning_params = {'n-estimators': IntegerParameter(10, 100),
                'min-samples-leaf': IntegerParameter(100, 200)}
objective_metric_name = "rootmse"
metric_definitions = [{"Name": "rootmse", "Regex": "rootmse: ([0-9\\.]+)"}]

In [19]:
# and fitting the tuner
tuner_rf = HyperparameterTuner(
    sklearn,
    objective_metric_name,
    tuning_params,
    metric_definitions,
    objective_type="Minimize",
    max_jobs=40,
    max_parallel_jobs=10,
)
tuner_rf.fit(s3_input)

......................................................................................................................................................................................................................................!


Before deploying the best random forest estimator, I will also tune the LightGBM model. Then I will deploy both at once, so that I can save a bit of resources by not having a random forest predictor run idle while the LGBM tuning is happening.

## 5. LightGBM

The final step of training will be using the previously defined custom container to train the LightGBM model. Once it is done, I can compare the prediction results of both random forest and LightGBM, and then the predictions when they are combined. After that, and before the summary, I will add a very short section that explains how the model can be used to compare different outcomes for a given customer with/without a campaign.

In [20]:
# get the data without outliers - technically not necessary as the dataset will be the same as for RF
# that is, as long as the seed is set
train_name = 'data/starbucks_train_lgb.csv'
test_name = 'data/starbucks_test_lgb.csv'

Xtrain_lg, Xtest_lg = split_data(df[df['hourly_sum'] <= df['hourly_sum'].quantile(0.95)], 
                                 features_lgbm, standardize=False)

Xtrain_lg = pd.concat([Xtrain_lg.iloc[:,-1], Xtrain_lg.iloc[:,:-1]], axis=1)
Xtest_lg = pd.concat([Xtest_lg.iloc[:,-1], Xtest_lg.iloc[:,:-1]], axis=1)

Xtrain_lg.to_csv(train_name, index=False)
Xtest_lg.to_csv(test_name, index=False)

# and upload it to S3
train_location = sess.upload_data(path=train_name, 
                                  bucket=bucket,
                                  key_prefix='lgbm')

test_location = sess.upload_data(path=test_name, 
                                 bucket=bucket,
                                 key_prefix='lgbm')

# where in S3 the data is located
s3_input = {'training': train_location, 'validating': test_location}

Now that I have the data prepared for LightGBM, I need to define parameters for a tuning job, as well as the metric I am optimizing towards.

In [21]:
# default parameters
set_hypers={'boosting_type': 'gbdt',
    'objective': 'regression',
    'metric': 'rmse',
    'feature_fraction': 1,
    'verbose': 0,
    'learning_rate': 0.01}

# parameters to tune
tuning_params = {'num_leaves': IntegerParameter(12, 24),
                'max_depth': IntegerParameter(4, 7)}

# metric definition
objective_metric_name = "rootmse"
metric_definitions = [{"Name": "rootmse", "Regex": "rootmse: ([0-9\\.]+)"}]

# estimator definition that uses the definitions above, as well as the LightGBM docker container
lgb = sagemaker.estimator.Estimator(container_image_uri,
                       role, 1, 'ml.c4.xlarge',
                       output_path="s3://{}/output".format(sess.default_bucket()),
                       sagemaker_session=sess,
                       hyperparameters=set_hypers)

In [22]:
tuner_lg = HyperparameterTuner(
    lgb,
    objective_metric_name,
    tuning_params,
    metric_definitions,
    objective_type="Minimize",
    max_jobs=40,
    max_parallel_jobs=10,
)
tuner_lg.fit(s3_input)

..............................................................................................................................................................................................................................!


In [23]:
predictor_lg = tuner_lg.deploy(initial_instance_count=1, instance_type="ml.m5.xlarge", serializer=csv_serializer)


2022-01-10 17:48:37 Starting - Preparing the instances for training
2022-01-10 17:48:37 Downloading - Downloading input data
2022-01-10 17:48:37 Training - Training image download completed. Training in progress.
2022-01-10 17:48:37 Uploading - Uploading generated training model
2022-01-10 17:48:37 Completed - Training job completed
----!

In [24]:
# deploying the best estimator for RF
# bestrf = SKLearn.attach(tuner_rf.best_training_job())
predictor_rf = tuner_rf.deploy(initial_instance_count=1, instance_type="ml.m5.xlarge")


2022-01-10 17:28:51 Starting - Preparing the instances for training
2022-01-10 17:28:51 Downloading - Downloading input data
2022-01-10 17:28:51 Training - Training image download completed. Training in progress.
2022-01-10 17:28:51 Uploading - Uploading generated training model
2022-01-10 17:28:51 Completed - Training job completed
-----!

Both predictors are now deployed, now I can check how they would perform separately and together. I will still use RMSE as the evaluation metric, since it is a proper metric for regression tasks.

## 6. Predictions

In [25]:
# and we can make the predictions
# first on random forest
preds_rf = predictor_rf.predict(Xtest_rf.iloc[:,1:])

# then on lightgbm
data_to_predict = pd.read_csv(test_name, header=None)
data_to_predict = data_to_predict.iloc[:,1:]
preds_lgb = predictor_lg.predict(data_to_predict.values).decode("utf-8")
preds_lgb = preds_lgb.split('\n')
preds_lgb = [float(elem) for elem in preds_lgb[:-1]]

# and then combined
preds_combined = (preds_rf + preds_lgb) / 2

rmse_rf = mean_squared_error(Xtest_rf.iloc[:,0], preds_rf, squared=False)
rmse_lgb = mean_squared_error(Xtest_lg.iloc[:,0], preds_lgb, squared=False)
rmse_comb = mean_squared_error(Xtest_lg.iloc[:,0], preds_combined, squared=False)

print('RMSE for random forests: ' + str(rmse_rf))
print('RMSE for LightGBM: ' + str(rmse_lgb))
print('RMSE for combined predictions: ' + str(rmse_comb))


The csv_serializer has been renamed in sagemaker>=2.
See: https://sagemaker.readthedocs.io/en/stable/v2.html for details.


RMSE for random forests: 0.513866505299603
RMSE for LightGBM: 0.5127030740376889
RMSE for combined predictions: 0.5129096102976082


We can see that the predictions from random forests and LGBM are both slightly better than from Linear Regression. The  RMSE value that can be obtained when the predictions are mixed does not get better. Moreover, RMSE is still quite high. Ideas about how to approach it will be listed in the summary, and now, let's have a look at how we can use the models to assess the effect of campaigns on a specific person.

## 7. How to analyse campaigns

In order to analyse the impact of campaigns on a specific customer, let's first choose one of them. After that, we can build a few different data points, each for the same customer but with a different campaign, and push them to the endpoint. Once we have the predictions, we can analyze the results.

In [26]:
# let's check what the columns are for the dataset
Xtest_lg.head()

,hourly_sum,reward,difficulty,channels_int,offer_type_int,gender,age,income,year
31422,0.519747,5.0,20.0,1.0,1.0,1,54,73000.0,2017
33845,0.263431,0.0,0.0,999.0,999.0,0,49,94000.0,2016
24706,0.056657,0.0,0.0,999.0,999.0,0,70,73000.0,2017
27680,0.112018,0.0,0.0,999.0,999.0,1,79,101000.0,2016
20362,0.009596,0.0,0.0,999.0,999.0,1,75,71000.0,2015


In [27]:
# later on I'll take the second person 
profile = pd.read_json('data/profile.json', orient='records', lines=True)
profile = clean_profile(profile)
profile.head()

,gender,age,id,became_member_on,income,year
0,0,55,0610b486422d4921ae7d2bf64640c50b,20170715,112000.0,2017
1,0,75,78afa995795e4d85b5d9ceeca43f5fef,20170509,100000.0,2017
2,1,68,e2127556f4f64592b11af22de27a7932,20180426,70000.0,2018
3,1,65,389bc3fa690240e798340f5a15918d5c,20180209,53000.0,2018
4,1,58,2eeac8d8feae4a8cad5a6af0499a211d,20171111,51000.0,2017


In [28]:
# we can predict on ALL the campaigns, so all I need is the right columns for the predictor
portfolio = pd.read_json('data/portfolio.json', orient='records', lines=True)
portfolio = clean_portfolio(portfolio)
portfolio.head()

,reward,channels,difficulty,duration,offer_type,id,email,mobile,social,web,bogo,discount,informational,channels_int,offer_type_int,duration_h
0,10,"[email, mobile, social]",10,7,bogo,ae264e3637204a6fb9bb56bc8210ddfd,1,1,1,0,1,0,0,0,0,168
1,10,"[web, email, mobile, social]",10,5,bogo,4d5c57ea9a6940dd891ad53e9dbe8da0,1,1,1,1,1,0,0,3,0,120
2,0,"[web, email, mobile]",0,4,informational,3f207df678b143eea3cee63160fa8bed,1,1,0,1,0,0,1,2,2,96
3,5,"[web, email, mobile]",5,7,bogo,9b98b8c7a33c4b65b9aebfe6a799e6d9,1,1,0,1,1,0,0,2,0,168
4,5,"[web, email]",20,10,discount,0b1e1539f2cc45b7b9fa7c272da2e1d7,1,0,0,1,0,1,0,1,1,240


In [67]:
# suppress settingwithcopywarning
pd.options.mode.chained_assignment = None

def get_one_customer(po_df, pr_df, n):
    # get the right columns from portfolio
    df_onecustomer_analysis = po_df[['reward', 'difficulty', 'channels_int', 'offer_type_int']]

    # add columns for the nth person in the profile dataset
    df_onecustomer_analysis['gender'] = pr_df.at[n, 'gender']
    df_onecustomer_analysis['age'] = pr_df.at[n, 'age']
    df_onecustomer_analysis['income'] = pr_df.at[n, 'income']
    df_onecustomer_analysis['year'] = pr_df.at[n, 'year']

    # one row FOR THE NO CAMPAIGN option
    no_campaign = [0,0, 999, 999, 
                   pr_df.at[n, 'gender'], 
                   pr_df.at[n, 'age'], 
                   pr_df.at[n, 'income'], 
                   pr_df.at[n, 'year']]
    df_length = len(df_onecustomer_analysis)
    
    # add the row
    df_onecustomer_analysis.loc[df_length] = no_campaign
    return df_onecustomer_analysis

In [68]:
df_one = get_one_customer(portfolio, profile, 1)
df_one.to_csv('data/test.csv', index=False)
df_one = pd.read_csv('data/test.csv', header=None)
preds_lgb_t = predictor_lg.predict(df_one.values).decode("utf-8")
preds_lgb_t = preds_lgb_t.split('\n')
preds_lgb_t = [float(elem) for elem in preds_lgb_t[:-1]]
headers = df_one.iloc[0]
df_one  = pd.DataFrame(df_one.values[1:], columns=headers)
df_one['hourly_predicted'] = preds_lgb_t
df_one

The csv_serializer has been renamed in sagemaker>=2.
See: https://sagemaker.readthedocs.io/en/stable/v2.html for details.


,reward,difficulty,channels_int,offer_type_int,gender,age,income,year,hourly_predicted
0,10,10,0,0,0,75,100000.0,2017,0.851333
1,10,10,3,0,0,75,100000.0,2017,0.865740
2,0,0,2,2,0,75,100000.0,2017,0.314531
3,5,5,2,0,0,75,100000.0,2017,0.830446
4,5,20,1,1,0,75,100000.0,2017,0.851840
5,3,7,3,1,0,75,100000.0,2017,0.870468
6,2,10,3,1,0,75,100000.0,2017,0.871840
7,0,0,0,2,0,75,100000.0,2017,0.317760
8,5,5,3,0,0,75,100000.0,2017,0.864012
9,2,10,2,1,0,75,100000.0,2017,0.816603


As we can see, the customer with index 1 has the highest hourly stake for campaigns with index 5 and 6. It is also quite visible that the informative campaigns (index 2 and 7) have a much lower hourly stake, although, they also do not need any specific reward, so they can be ran as long as the hourly cost of running such a campaign per customer is lower than 10 cents (as the hourly spend for periods without any campaign is 0.21 cents per hour).

One question that remains unanswered with this model is what offer is best to send in case given different rewards. Short of building a different model that could answer this question better, we could use a simple heuristic - it is better to always go for an offer that has a lower reward with a similar spend. And if the no-offer spend is similar to offer spends, it is better to just not send an offer at all to the customer.

One shortcoming of the model is that it does not provide information on cases when the customers only make a purchase to get a bonus - but that could be done with a different model that picks up those bonus abusers.

Another example of a different customer is shown below: this customer also prefers discount campaigns, but of a different kind.

In [69]:
df_one = get_one_customer(portfolio, profile, 2)
df_one.to_csv('data/test.csv', index=False)
df_one = pd.read_csv('data/test.csv', header=None)
preds_lgb_t = predictor_lg.predict(df_one.values).decode("utf-8")
preds_lgb_t = preds_lgb_t.split('\n')
preds_lgb_t = [float(elem) for elem in preds_lgb_t[:-1]]
headers = df_one.iloc[0]
df_one  = pd.DataFrame(df_one.values[1:], columns=headers)
df_one['hourly_predicted'] = preds_lgb_t
display(df_one)

The csv_serializer has been renamed in sagemaker>=2.
See: https://sagemaker.readthedocs.io/en/stable/v2.html for details.


,reward,difficulty,channels_int,offer_type_int,gender,age,income,year,hourly_predicted
0,10,10,0,0,1,68,70000.0,2018,0.216822
1,10,10,3,0,1,68,70000.0,2018,0.217526
2,0,0,2,2,1,68,70000.0,2018,0.104176
3,5,5,2,0,1,68,70000.0,2018,0.300197
4,5,20,1,1,1,68,70000.0,2018,0.366113
5,3,7,3,1,1,68,70000.0,2018,0.271602
6,2,10,3,1,1,68,70000.0,2018,0.294499
7,0,0,0,2,1,68,70000.0,2018,0.092322
8,5,5,3,0,1,68,70000.0,2018,0.234265
9,2,10,2,1,1,68,70000.0,2018,0.358298


## 8. Summary

In this notebook I looked at different models to train the Starbucks dataset. The main problem with the approach I took was the number of outliers that were affecting the predictions. If the training was done on the whole dataset, RMSE was much higher than acceptable (even with tree-based models) - getting rid of outliers helped immensely.

However, the outliers are valid data points that are composed of people who spend a lot soon after seeing the offer. As a result, they should be included in any model that would be put in production. One way how to handle them would be to create a <b> separate </b> model that tries to pick up customers who make a big purchase on the time of the offer viewing. Then, only people who are not purchasing immediately would be dealt with through the model I presented here.

Moreover, more data would be good to have - after getting rid of overlapping offers the dataset is much smaller; in addition, a lot of offers are never viewed and cannot be used as such in the training I performed.

Finally, some extensions of the model can be thought of that would include overlaps. Maybe there is a synergic effect of certain offers, or maybe some customers will actually spend less if they get too many offers at once (due to the feeling of being 'spammed'.

As such, the tree based models performed better than the linear regression model, but further experimentation would be needed.

In [32]:
# # clean-up
# predictor_rf.delete_endpoint()
# predictor_lg.delete_endpoint()

In [74]:
!tar chvfz notebook.tar.gz ../*

tar: Removing leading `../' from member names
../final_project/
../final_project/model.ipynb
../final_project/.ipynb_checkpoints/
../final_project/.ipynb_checkpoints/model-checkpoint.ipynb
../final_project/.ipynb_checkpoints/ExploratoryDataAnalysis-checkpoint.ipynb
../final_project/skscripts/
../final_project/skscripts/sk-train-lr.py
../final_project/skscripts/sk-train-rf.py
../final_project/data/
../final_project/data/portfolio.json
../final_project/data/profile.json
../final_project/data/starbucks_train_lgb.csv
../final_project/data/starbucks_test_lgb.csv
../final_project/data/.ipynb_checkpoints/
../final_project/data/test.csv
../final_project/data/starbucks_test_rf.csv
../final_project/data/starbucks_train_lr.csv
../final_project/data/starbucks_train_rf.csv
../final_project/data/transcript.json
../final_project/data/starbucks_train_lr_noout.csv
../final_project/data/finaldata.csv
../final_project/processing/
../final_project/processing/.ipynb_checkpoints/
../final_project/processing

../Project_Plagiarism_Detection/data/orig_taskd.txt
../Project_Plagiarism_Detection/data/g0pC_taske.txt
../Project_Plagiarism_Detection/data/g2pE_taskb.txt
../Project_Plagiarism_Detection/data/g0pE_taskc.txt
../Project_Plagiarism_Detection/data/g2pC_taskd.txt
../Project_Plagiarism_Detection/data/g1pB_taskd.txt
../Project_Plagiarism_Detection/data/g4pE_taskc.txt
../Project_Plagiarism_Detection/data/g3pA_taske.txt
../Project_Plagiarism_Detection/data/g2pC_taska.txt
../Project_Plagiarism_Detection/data/g2pE_taske.txt
../Project_Plagiarism_Detection/data/g2pE_taskd.txt
../Project_Plagiarism_Detection/data/g0pA_taskc.txt
../Project_Plagiarism_Detection/data/g4pE_taskb.txt
../Project_Plagiarism_Detection/data/g2pA_taskc.txt
../Project_Plagiarism_Detection/data/g4pB_taska.txt
../Project_Plagiarism_Detection/data/g0pB_taskc.txt
../Project_Plagiarism_Detection/data/g4pD_taskb.txt
../Project_Plagiarism_Detection/data/g3pA_taskc.txt
../Project_Plagiarism_Detection/data/g3pC_taskb.txt
../Project_P

../Project_Plagiarism_Detection/__MACOSX/data/._g0pD_taskb.txt
../Project_Plagiarism_Detection/helpers.py
../Project_Plagiarism_Detection/data.zip.1
../Project_Plagiarism_Detection/data.zip
../Project_Plagiarism_Detection/source_sklearn/
../Project_Plagiarism_Detection/source_sklearn/.ipynb_checkpoints/
../Project_Plagiarism_Detection/source_sklearn/.ipynb_checkpoints/train-checkpoint.py
../Project_Plagiarism_Detection/source_sklearn/train.py
../Project_Plagiarism_Detection/plagiarism_data/
../Project_Plagiarism_Detection/plagiarism_data/train.csv
../Project_Plagiarism_Detection/plagiarism_data/test.csv
../Project_Plagiarism_Detection/1_Data_Exploration.ipynb
../README.md
../Time_Series_Forecasting/
../Time_Series_Forecasting/txt_preprocessing.py
../Time_Series_Forecasting/household_power_consumption.txt
^C
